In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms

In [2]:
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

images, labels = next(iter(train_loader))
print(images.shape)
print(labels.shape)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:03<00:00, 2.81MB/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 110kB/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:01<00:00, 865kB/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 2.30MB/s]

Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw

torch.Size([64, 1, 28, 28])
torch.Size([64])


In [3]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128), # [64, 1, 28, 28] → [64, 784]
    nn.ReLU(),
    nn.Linear(128, 10)
)

In [4]:
loss_fn = nn.CrossEntropyLoss()
images, labels = next(iter(train_loader))
output = model(images)
loss = loss_fn(output, labels)
print(loss)

tensor(2.3021, grad_fn=<NllLossBackward0>)


In [5]:
lr = 0.01
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8

v = []
s = []
for param in model.parameters():
    v.append(torch.zeros_like(param))
    s.append(torch.zeros_like(param))


In [6]:
t = 0
for epoch in range(3):
    for images, labels in train_loader:
        # 1. Forward pass
        output = model(images)
        loss = loss_fn(output, labels)

        # 2. Backward pass
        model.zero_grad()
        loss.backward()

        # 3. Adam update
        with torch.no_grad():
            for i, param in enumerate(model.parameters()):
                t += 1
                grad = param.grad
                v[i] = beta1 * v[i] + (1-beta1) * grad
                s[i] = beta2 * s[i] + (1-beta2) * grad * grad
                v_corrected = v[i]/(1-beta1**t)
                s_corrected = s[i] / (1 - beta2**t)
                param.sub_(lr * v_corrected / (torch.sqrt(s_corrected) + epsilon))
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 1, Loss: 0.5474
Epoch 2, Loss: 0.1168
Epoch 3, Loss: 0.0641


In [7]:
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        output = model(images)
        predicted = torch.argmax(output, dim=1)
        for j in range(len(labels)):
            total += 1
            if predicted[j] == labels[j]:
                correct += 1

print(f"Accuracy: {correct / total * 100:.2f}%")

Accuracy: 96.31%


In [8]:
model2 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)
optimizer = torch.optim.Adam(model2.parameters(), lr=0.001)
for epoch in range(3):
    for images, labels in train_loader:
        output = model2(images)
        loss = loss_fn(output, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        output = model2(images)
        predicted = torch.argmax(output, dim=1)
        for j in range(len(labels)):
            total += 1
            if predicted[j] == labels[j]:
                correct += 1

print(f"Accuracy: {correct / total * 100:.2f}%")

Accuracy: 96.86%
